# A stranger's replication of the headline claims

Everything the site claims about itself should be recomputable from what
the site publishes. This notebook does that recomputation with nothing but
the published files -- the codebook and the data directory -- and the
standard scientific Python a stranger already has. It never imports the
pipeline; if a number here stops matching, either the method changed
without the codebook changing, or the codebook was never sufficient. Both
are defects, and the asserts below turn either one into a loud failure.

Every number in this document is printed by the code, never typed into the
prose: a claim you cannot recompute is a claim you are being asked to take
on faith, and this notebook is the part of the site that refuses to ask
for any.

In [ ]:
import csv
import json
import re
from datetime import date, timedelta
from pathlib import Path

import numpy as np

# Locate the checkout from wherever this notebook is run: the repository is
# the nearest ancestor that carries the published site data.
for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "docs" / "data" / "history.csv").exists():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError("run this notebook from inside the repository checkout")

DATA = ROOT / "docs" / "data"
CODEBOOK = DATA.parent / "codebook.md"

# The channel names come from the published file's own header, not from this
# notebook's memory of them.
with (DATA / "history.csv").open(encoding="utf-8") as fh:
    header = next(csv.reader(fh))
CHANNELS = [c for c in header if c not in ("date", "composite")]
print("channels, per the published header:", CHANNELS)

## A. The daily composite, rebuilt from the published shares

The codebook states the transform: each channel's day is scored as a
percentile of that day's share against the channel's own trailing window,
and the composite is the unweighted mean across channels. The window
length, the observation floor, and the tie, window-basis, and rounding
conventions are parsed out of the codebook's published text below rather
than restated here, so the notebook fails the day the codebook stops
saying enough for a stranger to reproduce the series.

In [ ]:
# The transform's constants, read from the codebook rather than remembered.
codebook_text = CODEBOOK.read_text(encoding="utf-8")

stated_constants = re.search(
    r"trailing (\d+) days,\s*minimum (\d+) observations", codebook_text)
assert stated_constants, (
    "the codebook no longer states the window length and the observation "
    "floor; a stranger cannot rebuild the series without them")
WINDOW_DAYS = int(stated_constants.group(1))
MIN_OBS = int(stated_constants.group(2))

for phrase, why in [
    ("calendar days", "whether the window counts calendar days or observations"),
    ("at or below", "the percentile convention at ties"),
    ("half-to-even", "how values exactly half-way between decimals round"),
]:
    assert phrase in codebook_text, f"the codebook no longer pins {why}"

print("window, in calendar days:", WINDOW_DAYS)
print("minimum trailing observations:", MIN_OBS)

In [ ]:
# The published quantity (shares.csv) and the published result (history.csv).
shares = {ch: {} for ch in CHANNELS}
with (DATA / "shares.csv").open(encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        day = date.fromisoformat(row["date"])
        for ch in CHANNELS:
            raw = (row.get(ch) or "").strip()
            if raw:
                shares[ch][day] = float(raw)

published = {}
n_published_values = 0
decimal_places = set()
with (DATA / "history.csv").open(encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        day = date.fromisoformat(row["date"])
        row_values = {}
        for col in [*CHANNELS, "composite"]:
            raw = (row.get(col) or "").strip()
            if raw:
                row_values[col] = float(raw)
                n_published_values += 1
                decimal_places.add(len(raw.rsplit(".", 1)[1]) if "." in raw else 0)
        published[day] = row_values

# Agreement is judged at the precision the site actually publishes, and that
# precision is the file's own, not this notebook's opinion of it.
assert len(decimal_places) == 1, (
    "history.csv no longer publishes one fixed number of decimal places, "
    "so 'agreement at the published precision' has stopped being defined")
PRECISION = decimal_places.pop()

print("share days per channel:", {ch: len(shares[ch]) for ch in CHANNELS})
print("published days:", len(published), "| published values:", n_published_values)
print("published decimal places:", PRECISION)

In [ ]:
# Rebuild: each channel-day is the share's percentile -- counting ties as
# at-or-below, per the codebook -- against every observation in the trailing
# calendar window ending on the day itself, never using future data; the
# composite is the unweighted mean where every channel is scored.
scores = {}
for ch in CHANNELS:
    items = sorted(shares[ch].items())
    day_ordinals = np.array([d.toordinal() for d, _ in items])
    values = np.array([v for _, v in items])
    for i in range(len(items)):
        window_lo = day_ordinals[i] - (WINDOW_DAYS - 1)
        j = int(np.searchsorted(day_ordinals, window_lo, side="left"))
        window = values[j:i + 1]
        if len(window) < MIN_OBS:
            continue
        pctl = 100.0 * float(np.count_nonzero(window <= values[i])) / len(window)
        scores.setdefault(date.fromordinal(int(day_ordinals[i])), {})[ch] = pctl

for day_scores in scores.values():
    if len(day_scores) == len(CHANNELS):
        day_scores["composite"] = sum(day_scores[c] for c in CHANNELS) / len(CHANNELS)

# Compare over EVERY published value. A comparison that quietly skips the
# values it cannot rebuild is how an agreement rate stays reassuring while
# the series drifts, so the first assert is about coverage, not agreement.
n_compared = n_agree = 0
mismatches = []
for day, published_row in published.items():
    rebuilt = scores.get(day, {})
    for col, want in published_row.items():
        if col not in rebuilt:
            continue
        n_compared += 1
        if round(rebuilt[col], PRECISION) == want:
            n_agree += 1
        else:
            mismatches.append((str(day), col, want, round(rebuilt[col], PRECISION)))

print("compared:", n_compared, "of", n_published_values, "published values")
print("agree:", n_agree, "| mismatches:", mismatches[:5])
assert n_compared == n_published_values, (
    "the rebuild could not score every published value: the published "
    "series contains values its published inputs cannot explain")
assert n_agree == n_compared, (
    "the rebuild disagrees with the published history; either the pipeline "
    "changed without the codebook changing, or the codebook has stopped "
    "being sufficient to reproduce the series it documents")

## B. The registered hit rate and the base rates that make it honest

The validation payload registers a hit rate: the count of frozen validation
events with a detected same-channel episode inside the registered window.
The baselines payload publishes the context that makes that number
interpretable -- the strict start-based reading, the naive any-channel
detector, and the deterministic chance rate for randomly dated events. All
of them are recomputed here from the published episodes table and the
published event list, and compared to both payloads, event by event.

In [ ]:
# The frozen event list and the registered window, from the published
# validation payload; the detected episodes, from the published table.
validation = json.loads((DATA / "validation.json").read_text(encoding="utf-8"))
WINDOW = int(validation["hit_rate"]["window_days"])
events = [
    {"name": e["name"], "channel": e["channel"],
     "date": date.fromisoformat(e["date"]), "published_hit": bool(e["hit"])}
    for e in validation["hit_rate"]["episodes"]
]

episodes = []
with (DATA / "episodes.csv").open(encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        episodes.append({"channel": row["channel"],
                         "start": date.fromisoformat(row["start"]),
                         "end": date.fromisoformat(row["end"])})


def hit_registered(event):
    """The registered criterion: a same-channel episode day within the window."""
    return any(
        p["channel"] == event["channel"]
        and p["start"] - timedelta(days=WINDOW)
        <= event["date"]
        <= p["end"] + timedelta(days=WINDOW)
        for p in episodes)


def hit_strict_start(event):
    """The stricter reading: the episode START within the window."""
    return any(
        p["channel"] == event["channel"]
        and abs((p["start"] - event["date"]).days) <= WINDOW
        for p in episodes)


def hit_any_channel(event):
    """The naive detector: any episode in ANY channel nearby."""
    return any(
        p["start"] - timedelta(days=WINDOW)
        <= event["date"]
        <= p["end"] + timedelta(days=WINDOW)
        for p in episodes)


registered = sum(hit_registered(e) for e in events)
strict = sum(hit_strict_start(e) for e in events)
naive = sum(hit_any_channel(e) for e in events)

# The chance rate, exactly and deterministically: per channel, the fraction
# of days in the episode-covered span lying within the window of any
# episode, summed over the events' channels. No simulation, no seed.
span_start = min(p["start"] for p in episodes)
span_end = max(p["end"] for p in episodes)
span = [span_start + timedelta(days=i)
        for i in range((span_end - span_start).days + 1)]
covered_fraction = {}
for ch in {e["channel"] for e in events}:
    channel_episodes = [p for p in episodes if p["channel"] == ch]
    covered = sum(
        1 for d in span
        if any(p["start"] - timedelta(days=WINDOW) <= d <= p["end"] + timedelta(days=WINDOW)
               for p in channel_episodes))
    covered_fraction[ch] = covered / len(span)
chance = sum(covered_fraction[e["channel"]] for e in events)

print("events:", len(events), "| episodes:", len(episodes),
      "| window, days either side:", WINDOW)
print("registered hits:", registered, "| strict-start:", strict,
      "| naive any-channel:", naive, "| chance-expected:", round(chance, 1))

baselines = json.loads(
    (DATA / "detection_baselines.json").read_text(encoding="utf-8"))
context = baselines["hit_rate_context"]
overall = validation["hit_rate"]["overall"]
assert [hit_registered(e) for e in events] == [e["published_hit"] for e in events], (
    "the per-event hit flags in validation.json do not reproduce from "
    "episodes.csv under the registered criterion")
assert (len(events), registered) == (overall["n"], overall["hits"]), (
    "the registered hit rate in validation.json does not reproduce")
assert (registered, strict, naive, len(events), len(episodes)) == (
    context["registered_criterion_hits"], context["strict_start_hits"],
    context["naive_any_channel_hits"], context["n_events"],
    context["n_episodes"]), (
    "the hit-rate context in detection_baselines.json does not reproduce")
assert round(chance, 1) == context["chance_expected_hits"], (
    "the deterministic chance rate in detection_baselines.json does not "
    "reproduce")

## C. One vintage, rebuilt from the panel's recipe

The vintages panel claims the ALFRED property: any published vintage of the
series can be reconstructed exactly by truncating the current history at
the vintage's last observation, dropping the days that vintage had not
published, and applying its recorded diffs. This section follows that
recipe for the vintage with the most recorded diffs -- the hardest one --
and verifies the row count the panel states for it.

In [ ]:
panel = json.loads((DATA / "vintages_panel.json").read_text(encoding="utf-8"))
vintage = max(panel["vintages"], key=lambda v: (v["n_diff_days"], v["published"]))

current = {}
with (DATA / "history.csv").open(encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        current[row["date"]] = dict(row)

absent = set(vintage["absent_days"])
last_observation = vintage["last_observation"]
rebuilt_vintage = {d: dict(r) for d, r in current.items()
                   if d <= last_observation and d not in absent}
for d, changed in vintage["diffs"].items():
    assert d in rebuilt_vintage, (
        "a recorded diff day is missing from the truncated current series; "
        "the recipe no longer applies to the file it describes")
    rebuilt_vintage[d].update(changed)

print("vintage published:", vintage["published"],
      "| last observation:", last_observation)
print("rebuilt rows:", len(rebuilt_vintage), "| stated rows:", vintage["n_rows"],
      "| diff days applied:", len(vintage["diffs"]),
      "| absent days dropped:", len(absent))
assert len(vintage["diffs"]) == vintage["n_diff_days"], (
    "the panel's stated diff count disagrees with its own diff records")
assert len(rebuilt_vintage) == vintage["n_rows"], (
    "the reconstruction recipe does not yield the row count the panel "
    "states; the ALFRED property is broken for this vintage")

## D. One row of the negative-results register, recomputed live

The register's premise is that the unflattering numbers are computed by the
same pipeline as the flattering ones. That premise is only worth anything
if a stranger can recompute a row. This section takes the placebo-overlap
row -- the finding that placebo channels overlap real episodes far above
chance -- and re-derives both of its percentages from the published placebo
episodes and the published episode table, then requires the register's own
text to carry exactly those numbers.

In [ ]:
register = json.loads(
    (DATA / "negative_results.json").read_text(encoding="utf-8"))
placebo_rows = [f for f in register["findings"]
                if f["source"].endswith("detection_baselines.json")
                and "placebo" in f["finding"].lower()]
assert placebo_rows, (
    "the placebo-overlap row has left the negative-results register; this "
    "section no longer has a published claim to check")
row = placebo_rows[0]
print("register row:", row["number"])

stated = re.match(
    r"(\d+) of (\d+) placebo episodes \(([\d.]+)%\) overlap real ones; "
    r"duration-preserving random placement expects ([\d.]+)%",
    row["number"])
assert stated, (
    "the register row no longer states its numbers in the form it "
    "registered, so they can no longer be checked against a recomputation")

# Live recomputation from the published files: the observed overlap count,
# and the exact duration-preserving random-placement expectation -- every
# possible start for each placebo interval inside the study window, scored
# against the published geopolitical episode days.
placebo = validation["placebo"]
geopolitical_days = set()
for p in episodes:
    d = p["start"]
    while d <= p["end"]:
        geopolitical_days.add(d)
        d += timedelta(days=1)

study_start = min(date.fromisoformat(r["start"]) for r in placebo["episodes"])
study_end = max(date.fromisoformat(r["end"]) for r in placebo["episodes"])
study_days = (study_end - study_start).days + 1

observed = 0
placement_chance = []
for r in placebo["episodes"]:
    start = date.fromisoformat(r["start"])
    duration = (date.fromisoformat(r["end"]) - start).days + 1
    if any(start + timedelta(days=i) in geopolitical_days
           for i in range(duration)):
        observed += 1
    n_starts = study_days - duration + 1
    overlapping_starts = sum(
        1 for offset in range(n_starts)
        if any(study_start + timedelta(days=offset + i) in geopolitical_days
               for i in range(duration)))
    placement_chance.append(overlapping_starts / n_starts)

n_placebo = len(placebo["episodes"])
observed_pct = round(100.0 * observed / n_placebo, 1)
chance_pct = round(100.0 * sum(placement_chance) / n_placebo, 1)
print("live: observed overlaps", observed, "of", n_placebo,
      "| observed", observed_pct, "% | random placement", chance_pct, "%")

assert observed == placebo["n_overlapping"] and n_placebo == placebo["n_placebo_episodes"], (
    "the observed placebo overlap no longer reproduces from the published "
    "placebo and episode tables")
assert (int(stated.group(1)), int(stated.group(2))) == (observed, n_placebo), (
    "the register row's counts are not the live counts")
assert (float(stated.group(3)), float(stated.group(4))) == (observed_pct, chance_pct), (
    "the register row's percentages are not the live percentages; the "
    "register has started hand-typing its numbers")

## What a failure here means

Nothing in this notebook has an opinion. Each section is an arithmetic
consequence of files the site publishes, ending in an assert. A red run
means the site's numbers no longer follow from the site's documentation:
the codebook changed meaning, a payload drifted from its inputs, or a
recipe stopped being exact. The fix is never to edit this notebook until
it goes quiet; it is to make the published files sufficient again.

The test suite executes this notebook on every run, so the site cannot
keep a headline this notebook cannot rebuild.